# Notebook 06 — XGBoost Model

**PharmaLens: An Explainable AI Framework for Drug–Target Interaction Prediction**

---

## Objective

Train an **XGBoost** (eXtreme Gradient Boosting) model for DTI prediction.

**Why XGBoost?** XGBoost is a powerful nonlinear model for structured numerical features. It consistently ranks among the top-performing methods for tabular data and provides built-in feature importance. It serves as a strong comparison point against deep learning models.

We will:
1. Train XGBoost with early stopping
2. Evaluate on validation and test sets
3. Perform SHAP analysis for explainability
4. Compare with baseline models

In [ ]:
import os, sys, warnings, pickle, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import *
from src.evaluation.metrics import compute_regression_metrics
from src.evaluation.experiment_tracker import ExperimentTracker
from src.models.xgboost_model import XGBoostDTI
from src.features.drug_features import DESCRIPTOR_NAMES, get_feature_names
from src.features.protein_features import get_protein_feature_names

set_seed(42)
COLORS = set_plot_style()
ensure_dirs()
tracker = ExperimentTracker()
print("Setup complete ✓")

## 1. Load Features and Splits

In [ ]:
with open(PROCESSED_DATA_DIR / 'drug_features.pkl', 'rb') as f:
    drug_features = pickle.load(f)
with open(PROCESSED_DATA_DIR / 'protein_features.pkl', 'rb') as f:
    protein_features = pickle.load(f)

train_df = pd.read_csv(SPLITS_DIR / 'random_train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'random_val.csv')
test_df = pd.read_csv(SPLITS_DIR / 'random_test.csv')

# Build feature matrices (same function as Notebook 05)
def build_feature_matrix(df, drug_features, protein_features):
    X_list, y_list = [], []
    for _, row in df.iterrows():
        if row['drug_id'] in drug_features and row['target_id'] in protein_features:
            combined = np.concatenate([drug_features[row['drug_id']], protein_features[row['target_id']]])
            X_list.append(combined)
            y_list.append(row['score'])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

X_train, y_train = build_feature_matrix(train_df, drug_features, protein_features)
X_val, y_val = build_feature_matrix(val_df, drug_features, protein_features)
X_test, y_test = build_feature_matrix(test_df, drug_features, protein_features)

# Feature names for SHAP
feature_names = get_feature_names() + get_protein_feature_names()
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f"Total features: {len(feature_names)}")

## 2. Train XGBoost

In [ ]:
# Initialize and train
xgb_model = XGBoostDTI(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.01,
    reg_lambda=1.0,
    early_stopping_rounds=50,
    random_state=42,
)

xgb_model.fit(X_train, y_train, X_val, y_val, verbose=True)
print(f"\nTraining time: {xgb_model.training_time:.1f}s")
print(f"Best iteration: {xgb_model.best_iteration}")

## 3. Evaluation

In [ ]:
# Predict
t_start = time.time()
y_train_pred = xgb_model.predict(X_train)
y_val_pred = xgb_model.predict(X_val)
y_test_pred = xgb_model.predict(X_test)
inference_time = time.time() - t_start

# Metrics
train_metrics = compute_regression_metrics(y_train, y_train_pred)
val_metrics = compute_regression_metrics(y_val, y_val_pred)
test_metrics = compute_regression_metrics(y_test, y_test_pred)

print("XGBoost Results:")
for split_name, metrics in [('Train', train_metrics), ('Val', val_metrics), ('Test', test_metrics)]:
    print(f"  {split_name:6s} — RMSE: {metrics['rmse']:.4f}  MAE: {metrics['mae']:.4f}  "
          f"Pearson: {metrics['pearson']:.4f}  Spearman: {metrics['spearman']:.4f}  "
          f"R²: {metrics['r2']:.4f}  CI: {metrics['ci']:.4f}")

# Log experiment
tracker.log_experiment(
    model='XGBoost', dataset='KIBA',
    representation='descriptors+fingerprints+aac+dpc',
    split='random', seed=42,
    hyperparameters=xgb_model.get_hyperparams(),
    metrics=test_metrics,
    training_time=xgb_model.training_time,
    inference_time=inference_time,
)

In [ ]:
# Actual vs Predicted + Residual plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test, y_test_pred, alpha=0.15, s=5, color=MODEL_COLORS['XGBoost'])
lims = [min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())]
axes[0].plot(lims, lims, 'k--', alpha=0.5)
axes[0].set_xlabel('Actual KIBA Score')
axes[0].set_ylabel('Predicted KIBA Score')
axes[0].set_title(f'XGBoost: Actual vs Predicted\nRMSE={test_metrics["rmse"]:.4f}, r={test_metrics["pearson"]:.4f}', fontweight='bold')

residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuals, alpha=0.15, s=5, color=MODEL_COLORS['XGBoost'])
axes[1].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Predicted KIBA Score')
axes[1].set_ylabel('Residual')
axes[1].set_title('XGBoost: Residuals', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgboost_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. SHAP Explainability

**SHAP (SHapley Additive exPlanations)** provides both global and local explanations:
- **Global:** Which features are most important overall?
- **Local:** For a specific prediction, which features pushed the score up or down?

In [ ]:
# Compute SHAP values (using a subsample for speed)
explainer = shap.TreeExplainer(xgb_model.model)

# Use a subsample of test data for SHAP analysis
n_shap = min(2000, len(X_test))
X_shap = X_test[:n_shap]
shap_values = explainer.shap_values(X_shap)
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# Global SHAP Summary Plot
plt.figure(figsize=(12, 8))
# Use short feature names for readability
short_names = [n[:20] for n in feature_names[:X_shap.shape[1]]]
shap.summary_plot(shap_values, X_shap, feature_names=short_names, 
                  max_display=20, show=False)
plt.title('XGBoost — SHAP Feature Importance (Top 20)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgboost_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot (mean absolute SHAP value)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=short_names,
                  plot_type='bar', max_display=20, show=False)
plt.title('XGBoost — Mean |SHAP| Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgboost_shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Local explanation for a specific prediction
example_idx = 0
print(f"Example prediction #{example_idx}:")
print(f"  Actual score: {y_test[example_idx]:.4f}")
print(f"  Predicted score: {y_test_pred[example_idx]:.4f}")
print(f"  Top contributing features:")

# Get top features for this prediction
sv = shap_values[example_idx]
top_idx = np.argsort(np.abs(sv))[::-1][:10]
for idx in top_idx:
    name = feature_names[idx] if idx < len(feature_names) else f'feature_{idx}'
    direction = '↑' if sv[idx] > 0 else '↓'
    print(f"    {name[:30]:<30s} SHAP={sv[idx]:+.4f} {direction}")

## 5. Save Model

In [ ]:
# Save trained model
xgb_model.save(XGBOOST_MODELS_DIR / 'xgb_best.json')

# Save results
xgb_results = {
    'model': 'XGBoost',
    'hyperparameters': xgb_model.get_hyperparams(),
    'train_metrics': train_metrics,
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
    'training_time': xgb_model.training_time,
    'best_iteration': xgb_model.best_iteration,
}
tracker.save_model_results('XGBoost', xgb_results)
print("Model and results saved ✓")

## Observations

1. XGBoost should improve substantially over the linear baselines, demonstrating that nonlinear feature interactions matter for DTI prediction.

2. **SHAP analysis** reveals which molecular and protein features most influence predictions. This provides interpretability that deep learning models lack by default.

3. The most important features can be related back to known chemistry — e.g., molecular weight, LogP, and specific fingerprint bits may correspond to pharmacophoric patterns.

> **Note:** These feature importances describe what the model learned, not biological causality.

## Conclusion

XGBoost provides a strong non-linear baseline with built-in explainability via SHAP. Results are logged for comparison with deep learning models.

**Next:** [Notebook 07 — DeepDTA Model](./07_deeptda_model.ipynb)